# Cross-validation for model selection (LogReg vs RF vs XGBoost) and single held-out test evaluation

Workflow:
1. Recreate the same train/val/test split by pdb_id used elsewhere in the project (random_state=42), so thetest set is identical to the rest of the project and excluded from CV entirely.
2. Run `GroupKFold` (grouped by `pdb_id`) on the train+val comparing Logistic Regression, Random Forest, and XGBoost, each as a full sklearn pipelin with its own preprocessing, matching notebooks 3 and 7.
3. Pick the model with the best mean CV score
4. Refit the winning pipeline on the full train+val .
5. Evaluate once on the untouched test set,this is the only number that should be reported as final test performance.

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

# Drive path for Google Colab
#from google.colab import drive
#drive.mount('/content/drive')
#df = pd.read_parquet('/content/drive/MyDrive/SB/SB_Project/classification_ring/data/processed/data_ml.parquet')

# Local path
df = pd.read_parquet('classification_ring/data/processed/data_ml.parquet')

num_features = [
    's_rsa', 's_phi', 's_psi', 's_a1', 's_a2', 's_a3', 's_a4', 's_a5',
    't_rsa', 't_phi', 't_psi', 't_a1', 't_a2', 't_a3', 't_a4', 't_a5'
]
cat_features = ['s_ss8', 's_3di_letter', 't_ss8', 't_3di_letter']
feature_cols = num_features + cat_features
label_cols = ['HBOND', 'VDW', 'IONIC', 'PIPISTACK', 'PICATION', 'SSBOND', 'PIHBOND']

df[num_features] = df[num_features].astype('float32')
print(df.shape)

## Step 1: Recreate the same train/val/test split (same seed as notebooks 2 / 3 / 7)

Guarantees the test set is identical to the one already used elsewhere, and excluded from everything below.

In [ ]:
pdb_ids = df['pdb_id'].unique().to_numpy()
print(f"Total unique PDBs: {len(pdb_ids)}")

train_pdbs, temp_pdbs = train_test_split(pdb_ids, test_size=0.30, random_state=42)
val_pdbs, test_pdbs   = train_test_split(temp_pdbs, test_size=0.50, random_state=42)

trainval_pdbs = np.concatenate([train_pdbs, val_pdbs])

trainval_df = df[df['pdb_id'].isin(trainval_pdbs)].reset_index(drop=True)
test_df     = df[df['pdb_id'].isin(test_pdbs)].reset_index(drop=True)

print(f"Train+Val pool: {len(trainval_df)} rows, {len(trainval_pdbs)} PDBs  <- used for CV / model selection")
print(f"Test (held out, untouched until the end): {len(test_df)} rows, {len(test_pdbs)} PDBs")

del df
gc.collect()

In [ ]:
def make_preprocessor():
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ])


def make_logreg_pipeline():
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', MultiOutputClassifier(
            LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
        ))
    ])


def make_rf_pipeline():
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', MultiOutputClassifier(
            RandomForestClassifier(
                n_estimators=100, max_depth=20,
                class_weight='balanced', random_state=42, n_jobs=2
            )
        ))
    ])


def make_xgb_pipeline():
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', MultiOutputClassifier(
            xgb.XGBClassifier(
                n_estimators=150, max_depth=6, learning_rate=0.1,
                tree_method='hist', random_state=42, n_jobs=2
            )
        ))
    ])


candidate_builders = {
    'logreg': make_logreg_pipeline,
    'rf': make_rf_pipeline,
    'xgboost': make_xgb_pipeline,
}

In [ ]:
THRESHOLDS = np.arange(0.00, 1.01, 0.01)

def evaluate_predictions(Y_true_df, Y_proba_list, label_cols, thresholds=None):
    """
    If thresholds=None:
        Optimize the threshold for each class by maximizing MCC.
    Otherwise:
        Use the supplied threshold for each class.
    """
    out = {}

    if thresholds is None:
        thresholds = {}

    for i, label in enumerate(label_cols):

        y_true = Y_true_df[label].values
        proba = Y_proba_list[i]
        y_score = proba[:, 1] if proba.shape[1] > 1 else np.zeros_like(y_true)


        if label not in thresholds:

            best_threshold = 0.5
            best_mcc = -1

            for t in THRESHOLDS:
                y_pred = (y_score >= t).astype(int)
                mcc = matthews_corrcoef(y_true, y_pred)

                if mcc > best_mcc:
                    best_mcc = mcc
                    best_threshold = t

            thresholds[label] = best_threshold


        t = thresholds[label]
        y_pred = (y_score >= t).astype(int)

        out[f'{label}_threshold'] = t
        out[f'{label}_mcc'] = matthews_corrcoef(y_true, y_pred)
        out[f'{label}_bal_acc'] = balanced_accuracy_score(y_true, y_pred)
        out[f'{label}_f1'] = f1_score(y_true, y_pred, zero_division=0)

        try:
            out[f'{label}_ap'] = average_precision_score(y_true, y_score)
            out[f'{label}_auroc'] = roc_auc_score(y_true, y_score)
        except ValueError:
            out[f'{label}_ap'] = np.nan
            out[f'{label}_auroc'] = np.nan

    return out, thresholds

## Step 3: run GroupKFold CV for each candidate model (on train+val)

In [ ]:
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

X_full = trainval_df[feature_cols]
Y_full = trainval_df[label_cols]
groups = trainval_df['pdb_id'].values

all_fold_rows = []

for model_name, builder in candidate_builders.items():
    print(f"\n=== Model: {model_name} ===")
    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_full, Y_full, groups=groups)):
        X_train = X_full.iloc[train_idx]
        X_val   = X_full.iloc[val_idx]
        Y_train = Y_full.iloc[train_idx].reset_index(drop=True)
        Y_val   = Y_full.iloc[val_idx].reset_index(drop=True)

        pipeline = builder()
        pipeline.fit(X_train, Y_train)

        Y_proba = pipeline.predict_proba(X_val)
        # compute validation metrics at optimal thresholds
        metrics, fold_thresholds = evaluate_predictions(
            Y_val,
            Y_proba,
            label_cols
            )
        metrics["model"] = model_name
        metrics["fold"] = fold_idx
        all_fold_rows.append(metrics)

        print(f"  fold {fold_idx} done")

        del pipeline, X_train, X_val, Y_train, Y_val,  Y_proba
        gc.collect()

cv_results = pd.DataFrame(all_fold_rows)
cv_results

## Step 4: pick the best model by mean macro-MCC across folds

In [ ]:
mcc_cols = [c for c in cv_results.columns if c.endswith('_mcc')]
cv_results['macro_mcc'] = cv_results[mcc_cols].mean(axis=1)

model_summary = cv_results.groupby('model')['macro_mcc'].agg(['mean', 'std']).sort_values('mean', ascending=False)
print(model_summary)

best_model_name = model_summary.index[0]
best_cv = cv_results[cv_results["model"] == best_model_name]

final_thresholds = {
    label: best_cv[f"{label}_threshold"].mean()
    for label in label_cols
}

print("Final thresholds:")
print(final_thresholds)
print(f"\nBest model: {best_model_name}")

## Step 5: refit the winning model's pipeline on the full train+val pool

In [ ]:
final_pipeline = candidate_builders[best_model_name]()
final_pipeline.fit(X_full, Y_full)
print(f"Final '{best_model_name}' pipeline fit on full train+val pool.")

In [ ]:
X_test = test_df[feature_cols]
Y_test = test_df[label_cols].reset_index(drop=True)

Y_test_proba = final_pipeline.predict_proba(X_test)

test_metrics, _ = evaluate_predictions(
    Y_test,
    Y_test_proba,
    label_cols,
    thresholds=final_thresholds
)
test_metrics_df = pd.DataFrame([test_metrics])
test_metrics_df


In [ ]:

#out_dir = '/content/drive/MyDrive/SB/SB_Project/classification_ring/results'
out_dir = 'classification_ring/results'
os.makedirs(out_dir, exist_ok=True)

cv_results.to_csv(os.path.join(out_dir, 'model_selection_cv_per_fold.csv'), index=False)
model_summary.to_csv(os.path.join(out_dir, 'model_selection_cv_summary.csv'))
test_metrics_df.to_csv(os.path.join(out_dir, 'final_test_metrics.csv'), index=False)

print("Saved: per-fold CV results, model comparison summary, and final test metrics.")

In [ ]:
load_test_metrics = True

if load_test_metrics:
    out_dir = 'classification_ring/results'
    os.makedirs(out_dir, exist_ok=True)
    test_metrics_df = pd.read_csv(os.path.join(out_dir, 'final_test_metrics.csv'))
    test_metrics = test_metrics_df.iloc[0].to_dict()
    print("Loaded final test metrics from CSV.")

rows = []
for label in label_cols:
    rows.append({
        "Class": label,
        "Threshold": test_metrics[f"{label}_threshold"],
        "MCC": test_metrics[f"{label}_mcc"],
        "Bal. acc.": test_metrics[f"{label}_bal_acc"],
        "AP": test_metrics[f"{label}_ap"],
        "AUROC": test_metrics[f"{label}_auroc"],
    })

compact_df = pd.DataFrame(rows)

for _, r in compact_df.iterrows():
    print(
        rf"\texttt{{{r['Class']}}} & "
        rf"{r['Threshold']:.3f} & {r['MCC']:.3f} & {r['Bal. acc.']:.3f} & "
        rf"{r['AP']:.3f} & {r['AUROC']:.3f} \\"
    )